# Rescaling of a Spline

We are going to build a spline $f_{K_{0}\rightarrow K}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{K_{0}\rightarrow K}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{K_{0}\rightarrow K}[{k\bmod K}]\,\beta^{n}(x-\delta x-k)$ characterized by the arbitrary positive integer period $K\in{\mathbb{N}}+1,$ the arbitrary nonnegative polynomial degree $n\in{\mathbb{N}},$ the arbitrary delay $\delta x\in{\mathbb{R}},$ and parameterized by the vector ${\mathbf{c}}_{K_{0}\rightarrow K}=\left(c_{K_{0}\rightarrow K}[k]\right)_{k=0}^{K-1}$ of spline coefficients. We want $f_{K_{0}\rightarrow K}$ to be as close as possible to the nominal spline $f_{0}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{0}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{0}[{k\bmod K_{0}}]\,\beta^{n_{0}}(x-\delta x_{0}-k).$ More precisely, we want to find ${\mathbf{c}}_{K_{0}\rightarrow K}$ such that the least-squares criterion $J=\frac{1}{2}\,\int_{0}^{K_{0}}\,\left(f_{K_{0}\rightarrow K}(\frac{K}{K_{0}}\,x)-f_{0}(x)\right)^{2}\,{\mathrm{d}}x$ is minimized. 

We illustrate the solution to this problem in the figure below, where one period of the $K_{0}$-periodic $f_{0}$ is shown in gray, with a gray grid that aligns with the bottom ticks. At the same time, one period of the $K$-periodic solution $f_{K_{0}\rightarrow K}$ is shown in blue, with a blue grid that aligns with the top ticks. For the two curves, the samples at the integers are indicated by circles, and the knots shown as dots.

Finally, as validation of the optimality of our solution, we print a convolution-based quantity that ought to vanish, up to numerical accuracy.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Random periodic cubic spline
f0 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(10), degree = 3)

# Plot
def update_plot (
    period0 = 10,
    degree0 = 3,
    delay0 = 0.0,
    period = 8,
    degree = 1,
    delay = 0.0
):
    global f0

    # Update of the spline
    if f0.period != period0:
        f0 = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period0),
            degree = f0.degree
        )
    f0.degree = degree0
    f0.delay = delay0

    # Rescaling
    f = f0.rescaled_projected(period = period, degree = degree, delay = delay)

    # Plots
    (fig, ax) = plt.subplots()
    # Dynamic range
    image = {f0.image(), f.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    if isinstance(plotrange, sk.interval.Singleton):
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.5,
            plotrange.midpoint + 0.5
        ))
    else:
        plotrange = sk.interval.Closed((
            plotrange.midpoint - 0.55 * plotrange.diameter,
            plotrange.midpoint + 0.55 * plotrange.diameter
        ))

    # Spline at the nominal scale
    f0.plot(
        (fig, ax),
        plotpoints = 200 + 1,
        plotdomain = sk.interval.Closed((0.0, period0)),
        plotrange = plotrange,
        curve_fmt = "#e0e0e0",
        curve_lw = 7.0,
        curve_markerfmt = "ok",
        curvestem_linefmt = "None",
        knot_marker = ".",
        periodboundstem_linefmt = "None"
    )
    ax.set_xticks([k for k in range(period0 + 1)])
    ax.grid(axis = "x")

    # Rescaled spline
    axr = ax.twiny()
    axr.set_xlabel("", color = "C0")
    axr.tick_params("x", colors = "C0")
    f.plot(
        (fig, axr),
        plotpoints = 200 + 1,
        plotdomain = sk.interval.Closed((0.0, period)),
        plotrange = plotrange,
        curvestem_linefmt = "None",
        knot_marker = "o",
        knot_color = "C0",
        periodboundstem_linefmt = "None"
    )
    axr.set_xticks([k for k in range(period + 1)])
    axr.grid(axis = "x", lw = 0.5, color = "C0")
    axr.spines.right.set_visible(True)
    axr.spines.top.set_visible(True)

    # Final display
    plt.show()

    # Convolution-based scalar product
    g = math.gcd(period0, period)
    f0up = f0.upscaled(magnification = period // g)
    fup = f.upscaled(magnification = period0 // g)
    fupv = ~fup
    djc = (fupv * fup)(0) - (fupv * f0up)(0)
    display(Math(
        r"""
        \left(f_{{\left({0:}\rightarrow{1:}\right)\uparrow{2:}}}^{{\vee}}*
        f_{{\left({0:}\rightarrow{1:}\right)\uparrow{2:}}}\right)(0)-
        \left(f_{{\left({0:}\rightarrow{1:}\right)\uparrow{2:}}}^{{\vee}}*
        \left(f_{{0}}\right)_{{\uparrow{3}}}\right)(0)={4:.2E}
        """.format(period0, period, period0 // g, period // g, djc)
    ))

# Interaction
widgets.interactive(
    update_plot,
    period0 = (1, max_period),
    degree0 = (0, max_degree),
    delay0 = (-max_delay, max_delay, 0.05),
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay, 0.05)
)
